In [ ]:
import pathlib

import matplotlib.pyplot as plt
import pandas as pd

%matplotlib inline

## Contents

1. [Problem formulation](#1-problem-formulation)
2. [Data sources](#2-data-sources)
3. [Mathematical background](#3-mathematical-background)
4. [Pre-registered hypotheses and tests](#4-pre-registered-hypotheses-and-tests)
5. [Exploratory analysis](#5-exploratory-analysis)
6. [Discussion](#6-discussion)
7. [Reproducibility](#7-reproducibility)
8. [References](#8-references)

## 1. Problem formulation

### 1.1 What is Volcano Rush

Volcano Rush is a semi-cooperative strategy board game for 6-8 players. The group is stranded on a volcanic island and must complete missions to gather resources and build a boat before the volcano erupts; that shared escape is the cooperative layer. On top of it, players also compete individually for the most mission-success points. The game is asymmetric: every character has different abilities, and some have entirely different mechanics. This asymmetry is what makes it challenging to balance, and it is the problem this notebook investigates.

- Full rules: [`docs/game_rules.md`](../../docs/game_rules.md)
- Characters: [`docs/characters.md`](../../docs/characters.md)
- Missions: [`docs/missions.md`](../../docs/missions.md)
- Volcano cards: [`docs/volcano_cards.md`](../../docs/volcano_cards.md)
- Complication cards: [`docs/complication_cards.md`](../../docs/complication_cards.md)

### 1.2 Research question

The notebook investigates a single overarching question:

> Is *Volcano Rush* balanced across its three supported player counts (6, 7, 8) and its six character roles, and how does its design position it within the landscape of real cooperative board games on BoardGameGeek?

Character presence cannot be tested directly in this simulation, because the game rules force every character into every game (all six roles appear at 6 players; at 7 and 8 players the same six are present with one or two non-Craftsman roles duplicated at random). Character balance is therefore decomposed into two layers: whether *which* non-Craftsman is duplicated affects the team's win probability, and whether any character dominates the *personal scoring* layer. Combined with the player-count question and the cross-source comparison, this gives four narrower sub-questions, each of which will be turned into one or more pre-registered hypotheses in [`docs/hypotheses.md`](../../docs/hypotheses.md) and tested in Section 4:

1. Does the team's win probability depend on the player count?
2. At 7 and 8 players, does the team's win probability depend on which non-Craftsman role is duplicated?
3. Does any single character dominate personal scoring across all player counts?
4. How does Volcano Rush's design position it within the landscape of well-rated cooperative games on BoardGameGeek?

### 1.3 Why this matters

Game balance is a core design problem in any asymmetric strategy game. If one strategy always wins, the game becomes predictable and the replay value collapses; if one character has a structural advantage, the game feels unfair; if the set of mechanics and abilities is too narrow, experienced players quickly converge on a single optimal playstyle. For casual party games this is often acceptable, but strategy games are judged on the depth and fairness of their decision space.

*Volcano Rush* adds a less common twist: the group shares the win condition (escape the island before the volcano erupts), but one player is still crowned the scoring winner at the end. This semi-cooperative mix is unusual in the Bulgarian tabletop market the author is familiar with, and it makes balance a two-layered problem. A balanced Volcano Rush therefore needs:

1. Similar team win rates across the three supported player counts.
2. No single character dominating the personal scoring.
3. A simulated difficulty profile comparable to well-rated cooperative games on BoardGameGeek, so that the game is challenging without being punishing.

These three conditions are what Section 4 operationalizes into testable hypotheses.

### 1.4 Scope and assumptions

**What the simulation models.** The simulation engine follows the rules documented in [`docs/game_rules.md`](../../docs/game_rules.md) exactly. Each player is replaced by a rule-based agent driven by hand-coded heuristics:

- Each character has a preferred mission type. For example, the Fire Starter prefers fire missions because its ability is strongest there; the Sailor prefers boat missions.
- Agents see the current scores of the other players. When they detect a scoring leader, with probability 75% they deprioritize that player during participant selection, introducing a mild competitive pressure against the leader.
- Agents have limited lookahead: they do not plan more than one round ahead and do not model the volcano deck beyond its visible card count.

**What the simulation does not model.** The simulation cannot reproduce any aspect of real-world play that depends on human-to-human interaction. In particular, it excludes:

- Table talk and in-game negotiation.
- House rules or ad-hoc rule interpretations.
- Social dynamics such as dominant or reluctant players, alliances, or grudges.
- Learning effects across sessions: every simulated game starts from a fresh agent state.

These omissions are limitations to keep in mind throughout the notebook. Conclusions drawn from the simulation apply to the rule-based-agent setting, and their transfer to real playtest sessions should itself be treated as a hypothesis, not a given.

## 2. Data sources

Two independent sources are used. They are loaded and described here; no inference is done in this section.

In [8]:
repository_path = pathlib.Path.cwd().parents[1]
simulations_path = repository_path / "data" / "simulations"
bgg_path = repository_path / "data" / "bgg"

### 2.1 Source 1: Monte Carlo simulation

Generated by [`simulation_engine/`](../../simulation_engine) and exported to [`data/simulations/`](../../data/simulations). Run metadata is recorded in [`data/simulations/manifest.json`](../../data/simulations/manifest.json).

**Sampling plan.**

- 4,000 games per player count (6, 7, 8), for a total of 12,000 games.
- Base seed: `20260428`. The same base seed is used across all three player-count scenarios; the per-game seed is `base_seed + game_index`, so every game is individually reproducible.

**Relationship to the Math Concepts project.** The simulation engine itself is the same version used in the Math Concepts phase of the project, but the random seed and sample size are different: the Math Concepts runs used `BASE_SEED = 1` with 1,000 games per player count, while this export uses `BASE_SEED = 20260428` with 4,000 games per count. Because the engine is deterministic given a seed, the two datasets share no individual games, and any pattern first observed in the Math Concepts work is therefore tested here on fresh, non-overlapping randomness.

**Files available.**

- `games.csv`: one row per simulated game (game_id, seed, player count, outcome, rounds played, boat parts built and required, volcano cards remaining, a mission-failure counter).
- `game_characters.csv`: one row per (game, character) pair, with the character's final personal score and a copy of the game outcome for convenience.
- `game_resources.csv`: per-game resource usage (units consumed and mission failures caused by shortage), in long format over `WOOD`, `STONE`, `ROPE`.
- `game_tools.csv`: per-game tool state (repairs and failures while damaged), in long format over `KNIFE`, `VESSEL`.
- `character_contributions.csv`: per-character contribution breakdown (missions participated, boat missions participated, tools repaired, lesser-evil uses, requirement discounts used). **Win games only.**
- `manifest.json`: sampling plan, engine commit, and a full column-level schema for every table above.

In [9]:
simulation_games_data = pd.read_csv(simulations_path / "games.csv")
simulation_characters_data = pd.read_csv(simulations_path / "game_characters.csv")
simulation_resources_data = pd.read_csv(simulations_path / "game_resources.csv")
simulation_tools_data = pd.read_csv(simulations_path / "game_tools.csv")
simulation_contributions_data = pd.read_csv(simulations_path / "character_contributions.csv")

simulation_games_data.head()

,game_id,seed,player_count,outcome,rounds_played,boat_parts_built,boat_parts_required,volcano_cards_remaining,mission_failures_any_extra
0,296325ecc0d5ffe2,20260428,6,win,20,3,3,2,1
1,259a5d07d71f3473,20260429,6,win,20,3,3,1,0
2,74570438e9a72e0c,20260430,6,loss,13,2,3,0,0
3,6559fe61497155ac,20260431,6,win,15,3,3,4,0
4,5723fa7ec997a08b,20260432,6,loss,16,2,3,0,0


### 2.2 Source 2: BoardGameGeek

The second data source is the public **BoardGameGeek (BGG)** database, obtained from the Kaggle dataset *Board Games Database from BoardGameGeek* by threnjen ([kaggle page](https://www.kaggle.com/datasets/threnjen/board-games-database-from-boardgamegeek)). The snapshot covers roughly 21,925 games and around 18.9 million individual user ratings, split across nine CSV tables under [`data/bgg/`](../../data/bgg). The upstream schema is documented verbatim in [`data/bgg/bgg_data_documentation.txt`](../../data/bgg/bgg_data_documentation.txt). The local snapshot was downloaded on 2026-04-19.

**Why this is a meaningful second source.** The simulation data answers how Volcano Rush behaves under its own rules. BGG answers how real cooperative games, as experienced by tens of thousands of players, sit on the difficulty, rating, and player-count axes. Treating Volcano Rush's simulated profile as a single point in BGG's empirical distribution lets the BGG comparison in Section 4 ask whether the game is typical or an outlier among well-rated co-op games, using a sample that has nothing to do with the simulation itself.

**Files used in hypothesis tests.**

- `games.csv` (21,925 rows): one row per BGG game. The columns that matter for this project are `BayesAvgRating` and `AvgRating` (community rating), `GameWeight` (the 1-5 complexity score used as the difficulty proxy), `MinPlayers` / `MaxPlayers` / `BestPlayers` (player-count range), `NumUserRatings` (sample-reliability filter), and `YearPublished`.
- `mechanics.csv` (21,925 rows): one row per game, with a binary indicator column per mechanic. Two columns are directly relevant: `Cooperative Game` and `Semi-Cooperative Game`. Both are used to filter the population down to cooperative-flavoured games when forming the comparison group for Hypothesis 4.

**Files available but not used in the pre-registered tests.** The following are loaded only for the exploratory section (Section 5) if a specific question calls for them, and are listed here for transparency:

- `themes.csv`, `subcategories.csv`: binary indicators per theme and per subcategory.
- `ratings_distribution.csv`: 0.0-10.0 rating histogram per game.
- `user_ratings.csv` (around 18.9 million rows): raw per-user rating. Loaded only if user-level analysis becomes necessary, which the current hypothesis plan does not require.
- `artists_reduced.csv`, `designers_reduced.csv`, `publishers_reduced.csv`: creator-level tags, not relevant to the balance question.

In [10]:
bgg_games_data = pd.read_csv(bgg_path / "games.csv")
bgg_mechanics_data = pd.read_csv(bgg_path / "mechanics.csv")
bgg_themes_data = pd.read_csv(bgg_path / "themes.csv")

bgg_games_data.head()

,BGGId,Name,Description,YearPublished,GameWeight,AvgRating,BayesAvgRating,StdDev,MinPlayers,MaxPlayers,...,Rank:partygames,Rank:childrensgames,Cat:Thematic,Cat:Strategy,Cat:War,Cat:Family,Cat:CGS,Cat:Abstract,Cat:Party,Cat:Childrens
0,1,Die Macher,die macher game seven sequential political rac...,1986,4.3206,7.61428,7.10363,1.57979,3,5,...,21926,21926,0,1,0,0,0,0,0,0
1,2,Dragonmaster,dragonmaster tricktaking card game base old ga...,1981,1.9630,6.64537,5.78447,1.45440,3,4,...,21926,21926,0,1,0,0,0,0,0,0
2,3,Samurai,samurai set medieval japan player compete gain...,1998,2.4859,7.45601,7.23994,1.18227,2,4,...,21926,21926,0,1,0,0,0,0,0,0
3,4,Tal der Könige,triangular box luxurious large block tal der k...,1992,2.6667,6.60006,5.67954,1.23129,2,4,...,21926,21926,0,0,0,0,0,0,0,0
4,5,Acquire,acquire player strategically invest business t...,1964,2.5031,7.33861,7.14189,1.33583,2,6,...,21926,21926,0,1,0,0,0,0,0,0


### 2.3 Cleaning and validation

Steps to document here:

- Missing values (which columns, how handled).
- Duplicate or inconsistent rows.
- Filters applied (for example, restricting BGG to games with the `Cooperative Game` mechanic and average rating at least 7.0).
- Sanity checks (row counts after filtering, ranges for numeric columns).

In [11]:
def reformat_columns(columns):
    return (columns
            .str.replace(r':([a-z])', lambda m: m.group(1).upper(), regex = True)
            .str.replace(":", ""))

In [12]:
bgg_games_data.columns = reformat_columns(bgg_games_data.columns)

In [13]:
bgg_mechanics_data

,BGGId,Alliances,Area Majority / Influence,Auction/Bidding,Dice Rolling,Hand Management,Simultaneous Action Selection,Trick-taking,Hexagon Grid,Once-Per-Game Abilities,...,Contracts,Passed Action Token,King of the Hill,Action Retrieval,Force Commitment,Rondel,Automatic Resource Growth,Legacy Game,Dexterity,Physical
0,1,1,1,1,1,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,3,0,1,0,0,1,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
3,4,0,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21920,347146,0,0,1,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
21921,347521,0,1,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
21922,348955,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
21923,349131,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### 2.4 Descriptive statistics

Summary tables and distribution plots that orient the reader before any testing. No inference here.

In [14]:
# Descriptive summaries: win rate per player count, ratings distribution,
# game weight distribution, etc. Plots without conclusions.

## 3. Mathematical background

This section explains the machinery used later so the reader does not have to take any formula on faith.

### 3.1 Monte Carlo estimation

Why the average of many simulated games is a valid estimator of the true win probability; the standard error of a proportion; why sample size matters.

### 3.2 Confidence intervals

Wilson interval for proportions; bootstrap confidence intervals for statistics whose distribution is unknown; the correct reading of a 95% CI (*not* "the true value is inside with 95% probability").

### 3.3 Hypothesis testing

Null and alternative hypotheses; p-values; chi-square test of independence; likelihood-ratio tests; the Holm-Bonferroni correction for testing multiple hypotheses in the same family.

### 3.4 Logistic regression

Odds and odds ratios; interpretation of coefficients; how a binary outcome (win / loss) is modelled; what a confidence interval on an odds ratio means in practice.

## 4. Pre-registered hypotheses and tests

Each hypothesis from [`docs/hypotheses.md`](../../docs/hypotheses.md) is written up in its own notebook under [`hypotheses/`](hypotheses/), so each test can be run, reviewed, and iterated on in isolation. The Holm-Bonferroni correction across the seven tests is applied here in Section 4.8 once each per-hypothesis notebook has been run.

* [Hypothesis 1 - Player-count balance](hypotheses/1_hypothesis_player_count_balance.ipynb)
* [Hypothesis 2 - Duplication effect at 7 and 8 players](hypotheses/2_hypothesis_duplication_effect.ipynb)
* [Hypothesis 3 - Character dominance in personal scoring](hypotheses/3_hypothesis_character_dominance.ipynb)
* [Hypothesis 4 - Semi-cooperative vs fully cooperative ratings on BGG](hypotheses/4_hypothesis_semicoop_vs_coop.ipynb)
* [Hypothesis 5 - Large-group vs smaller-group cooperative ratings on BGG](hypotheses/5_hypothesis_large_vs_small_group.ipynb)
* [Hypothesis 6 - Rating spread across large- vs smaller-group co-op](hypotheses/6_hypothesis_rating_spread.ipynb)
* [Hypothesis 7 - Simulation-optimal player count vs BGG community sweet spot](hypotheses/7_hypothesis_optimal_player_count.ipynb)

### 4.8 Summary of results

Table with one row per hypothesis: test, statistic, p-value, Holm-adjusted threshold, confidence interval, decision. Built at the end, not along the way.

## 5. Exploratory analysis

Everything in this section is **exploratory**: observations and patterns noticed while working with the data that were not pre-registered. Each finding must end with the acknowledgement that confirming it would require a new experiment with its own pre-registered hypothesis.

Candidate angles:

- Correlations between simulated volcano-tracker dynamics and outcome.
- Distribution of rounds played on winning vs losing games.
- BGG ratings as a function of player count and weight, focused on cooperative games.
- Text or thematic patterns in BGG cooperative games whose profile is close to Volcano Rush's simulated one.

## 6. Discussion

### 6.1 What the evidence supports

Restate the findings in plain language, with effect sizes and confidence intervals. No new tests.

### 6.2 Limitations

Agent-strategy assumptions, simulation sample size, choice of BGG difficulty proxy, scope of the multiple-testing correction.

### 6.3 Threats to validity

Possible confounders, model mis-specification, selection effects introduced by the BGG rating filter.

### 6.4 Future work

Concrete next experiments: stronger agents (including reinforcement learning), more player-count rollouts, richer difficulty proxies on the BGG side.

## 7. Reproducibility

- Python environment: [`environment.yml`](../../environment.yml).
- Simulation seed and sample size: recorded in [`data/simulations/manifest.json`](../../data/simulations/manifest.json).
- BGG snapshot: Kaggle download date and URL in Section 8.
- Commit hash of the simulation-engine run used for the final numbers: to be pasted once the run is frozen.
- How to re-generate the simulation data and the figures: one command per step.

## 8. References

- BoardGameGeek dataset: https://www.kaggle.com/datasets/threnjen/board-games-database-from-boardgamegeek
- Internal pre-registration: [`docs/hypotheses.md`](../../docs/hypotheses.md)
- Internal rulebook: [`docs/game_rules.md`](../../docs/game_rules.md)
- Simulation engine source: [`simulation_engine/`](../../simulation_engine)
- Statistical methods references: textbooks and papers to be listed as they are used in Section 3.